In [1]:
# Install packages
# Cell 1 — install and import libs (run once)
# If you already have these packages, pip will skip re-installation quickly.

import sys
import subprocess

def pip_install(packages):
    for p in packages:
        try:
            __import__(p.split('==')[0])
        except Exception:
            print("Installing", p)
            subprocess.check_call([sys.executable, "-m", "pip", "install", p])

# recommended packages (adjust versions if you prefer)
pip_install([
    "earthengine-api",    # Earth Engine Python API
    "geemap",             # interactive mapping
    "geopandas",          # read shapefiles/geojson
    "pyogrio",            # faster shapefile IO (optional)
    "rasterio",           # optional, sometimes used
    "Pillow",             # image handling
    "reportlab",          # building PDF
])

# Now imports
import ee
import geemap
import geopandas as gpd
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import urllib.request
import ipywidgets as widgets

print("Imports done. geemap version:", geemap.__version__)


Installing earthengine-api
Installing Pillow
Imports done. geemap version: 0.35.3


In [2]:
# Authenticating of the GEE project to use
ee.Authenticate()
ee.Initialize(
    project='vegetation-monitoring-474607'
)
print("Earth Engine initialized.")

Earth Engine initialized.


In [3]:
# Cell 3 — Upload AOI widget + helpers
from ipywidgets import FileUpload
upload_widget = FileUpload(accept='', multiple=False, description="Upload AOI (.geojson or .zip shapefile)")

display(upload_widget)

import zipfile
import tempfile
from pathlib import Path

def save_uploaded_file(upload_widget):
    """
    Save the uploaded file from ipywidgets.FileUpload to a temporary directory.
    Works for both classic Jupyter and VS Code Notebook widget formats.
    """
    if not upload_widget.value:
        print("⚠️ No file uploaded.")
        return None

    # Case 1: classic Jupyter (dict)
    if isinstance(upload_widget.value, dict):
        uploaded = list(upload_widget.value.values())[0]
        name = uploaded["metadata"]["name"]
        content = uploaded["content"]

    # Case 2: VS Code / newer ipywidgets (tuple)
    elif isinstance(upload_widget.value, tuple):
        uploaded = upload_widget.value[0]
        name = uploaded["name"]
        content = uploaded["content"]

    else:
        raise ValueError(f"Unsupported upload_widget.value type: {type(upload_widget.value)}")

    # Save to temporary directory
    import tempfile, os
    out_path = os.path.join(tempfile.gettempdir(), name)
    with open(out_path, "wb") as f:
        f.write(content)
    print("✅ Saved uploaded file to:", out_path)
    return out_path


def load_aoi_to_ee(local_path):
    # Accept zipped shapefile (.zip) or .geojson/.json/.shp
    ext = Path(local_path).suffix.lower()
    if ext == ".zip":
        extract_dir = tempfile.mkdtemp()
        with zipfile.ZipFile(local_path, 'r') as z:
            z.extractall(extract_dir)
        # find a shapefile or geojson
        shp = None
        for f in os.listdir(extract_dir):
            if f.lower().endswith(".shp") or f.lower().endswith(".geojson") or f.lower().endswith(".json"):
                shp = os.path.join(extract_dir, f)
                break
        if shp is None:
            raise FileNotFoundError("No .shp or .geojson inside uploaded zip.")
        gdf = gpd.read_file(shp)
    else:
        gdf = gpd.read_file(local_path)

    # Ensure geometry valid and in EPSG:4326
    if gdf.crs is None:
        print("AOI has no CRS. Assuming EPSG:4326.")
        gdf = gdf.set_crs("EPSG:4326")
    else:
        gdf = gdf.to_crs(epsg=4326)

    print("AOI features:", len(gdf))
    # Convert to GeoJSON and to EE FeatureCollection
    ee_fc = geemap.geopandas_to_ee(gdf, geodesic=False)
    return ee_fc, gdf


FileUpload(value=(), description='Upload AOI (.geojson or .zip shapefile)')

In [4]:
local_path = save_uploaded_file(upload_widget)
aoi, gdf = load_aoi_to_ee(local_path)
print("AOI loaded to EE. FeatureCollection size:", aoi.size().getInfo())

✅ Saved uploaded file to: C:\Users\PC\AppData\Local\Temp\Ward1.geojson
AOI features: 1
AOI loaded to EE. FeatureCollection size: 1


In [5]:
# Cell 4 — Sentinel-2 preprocessing & index functions
# We're using COPERNICUS/S2_SR (surface reflectance)
S2_COLLECTION = "COPERNICUS/S2_SR_HARMONIZED"

def mask_s2_sr_clouds(image):
    """Mask clouds using QA60 band in S2_SR (bitmask: 1024|2048)."""
    qa = image.select('QA60')
    # Bits 10 and 11: clouds and cirrus
    cloud_bit_mask = (1 << 10) | (1 << 11)
    mask = qa.bitwiseAnd(cloud_bit_mask).eq(0)
    return image.updateMask(mask)

def scale_s2(image):
    """Scale S2 Surface Reflectance (S2_SR uses 0-10000). Convert to float reflectance 0-1."""
    bands = image.select(['B2','B4','B8'])
    return image.addBands(bands.divide(10000.0), overwrite=True).copyProperties(image, image.propertyNames())

def add_indices_s2(image):
    """Compute NDVI, EVI (and keep original scaled bands)."""
    # make sure bands B4, B8, B2 are present and scaled to 0-1
    nir = image.select('B8').rename('NIR')
    red = image.select('B4').rename('RED')
    blue = image.select('B2').rename('BLUE')

    ndvi = nir.subtract(red).divide(nir.add(red)).rename('NDVI')
    # EVI formula adapted for reflectance (no *10000); use L=1
    evi = nir.subtract(red).multiply(2.5).divide(nir.add(red.multiply(6)).subtract(blue.multiply(7.5)).add(1)).rename('EVI')

    return image.addBands([ndvi, evi])

# Quick test: print collection size before/after for a small date range (run after AOI is defined)
s2 = ee.ImageCollection(S2_COLLECTION).filterBounds(aoi).filterDate('2024-01-01', '2024-01-31')
print("S2 scenes in Jan:", s2.size().getInfo())


S2 scenes in Jan: 30


In [6]:
# Cell 5 — prepare S2 collection for given year, compute per-image VCI using yearly NDVI min/max
year = 2024
start_date = f"{year}-01-01"
end_date = f"{year}-12-31"

# Build collection: filter, cloud mask, scale, indices
s2_raw = ee.ImageCollection(S2_COLLECTION).filterBounds(aoi).filterDate(start_date, end_date)
s2_pre = (s2_raw
         .map(mask_s2_sr_clouds)
         .map(scale_s2)
         .map(add_indices_s2)
        )

print("Sentinel-2 collection size (preprocessed):", s2_pre.size().getInfo())

# Compute NDVI min and max across the full year (for VCI)
ndvi_min_img = s2_pre.select('NDVI').reduce(ee.Reducer.min()).rename('NDVI_min')
ndvi_max_img = s2_pre.select('NDVI').reduce(ee.Reducer.max()).rename('NDVI_max')

# Function to add VCI per image
def add_vci(img):
    ndvi = img.select('NDVI')
    vci = ndvi.subtract(ndvi_min_img).divide(ndvi_max_img.subtract(ndvi_min_img)).rename('VCI')
    return img.addBands(vci)

s2_indices = s2_pre.map(add_vci)
print("Bands in first image:", s2_indices.first().bandNames().getInfo())


Sentinel-2 collection size (preprocessed): 366
Bands in first image: ['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B9', 'B11', 'B12', 'AOT', 'WVP', 'SCL', 'TCI_R', 'TCI_G', 'TCI_B', 'MSK_CLDPRB', 'MSK_SNWPRB', 'QA10', 'QA20', 'QA60', 'MSK_CLASSI_OPAQUE', 'MSK_CLASSI_CIRRUS', 'MSK_CLASSI_SNOW_ICE', 'NDVI', 'EVI', 'VCI']


In [ ]:
## Cell 6 modified version
# Optimized Cell 6 — faster monthly composites and AOI stats extraction
def compute_monthly_stats_fast(collection, aoi, year=year, scale=10):
    """
    Compute monthly mean, min, max composites for NDVI, EVI, and VCI over AOI.
    Optimized to minimize Earth Engine calls.
    """
    rows = []
    print(f"📅 Computing monthly NDVI/EVI/VCI stats for {year}...")

    for m in range(1, 13):
        start = ee.Date.fromYMD(year, m, 1)
        end = start.advance(1, "month")
        coll = collection.filterDate(start, end)
        count = coll.size().getInfo()
        print(f"→ Month {m:02d}: {count} images")

        if count == 0:
            rows.append({
                "month": m,
                "mean_NDVI": None, "min_NDVI": None, "max_NDVI": None,
                "mean_EVI": None,  "min_EVI": None,  "max_EVI": None,
                "mean_VCI": None,  "min_VCI": None,  "max_VCI": None,
                "count": 0
            })
            continue

        # Precompute composites
        mean_img = coll.mean().clip(aoi)
        min_img = coll.min().clip(aoi)
        max_img = coll.max().clip(aoi)

        # Stack all composites together for single reduce
        stacked = mean_img.addBands(min_img).addBands(max_img)
        # Rename for clarity
        stacked = stacked.select(
            ["NDVI", "EVI", "VCI", "NDVI_1", "EVI_1", "VCI_1", "NDVI_2", "EVI_2", "VCI_2"],
            ["mean_NDVI", "mean_EVI", "mean_VCI", "min_NDVI", "min_EVI", "min_VCI", "max_NDVI", "max_EVI", "max_VCI"]
        )

        # Compute mean across AOI for all bands in one go
        stats = stacked.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=aoi.geometry() if isinstance(aoi, ee.FeatureCollection) else aoi,
            scale=scale,
            maxPixels=1e13
        ).getInfo()

        stats["month"] = m
        stats["count"] = count
        rows.append(stats)

    df = pd.DataFrame(rows)
    print("✅ Monthly AOI stats computed successfully.")
    display(df.head())
    return df


# --- Run the optimized function ---
df_monthly = compute_monthly_stats_fast(s2_indices, aoi, year=year, scale=10)

# Save results
os.makedirs("outputs", exist_ok=True)
csv_path = f"outputs/sentinel2_monthly_summary_{year}.csv"
df_monthly.to_csv(csv_path, index=False)
print(f"💾 Saved CSV: {csv_path}")


📅 Computing monthly NDVI/EVI/VCI stats for 2024...
→ Month 01: 30 images


In [9]:

def compute_monthly_stats_chunked(collection, aoi, year=2024, scale=250):#modified scale 10 >250
    """
    Efficient, memory-safe computation of monthly NDVI/EVI/VCI stats.
    Processes each month individually to avoid 'User memory limit exceeded'.
    """
    print(f"📅 Computing monthly NDVI/EVI/VCI stats for {year} (memory-safe)...")
    rows = []

    for m in range(1, 13):
        start = ee.Date.fromYMD(year, m, 1)
        end = start.advance(1, "month")
        coll = collection.filterDate(start, end)

        count = coll.size().getInfo()
        print(f"→ Month {m:02d}: {count} images")

        if count == 0:
            rows.append({
                "month": m,
                "mean_NDVI": None, "min_NDVI": None, "max_NDVI": None,
                "mean_EVI": None, "min_EVI": None, "max_EVI": None,
                "mean_VCI": None, "min_VCI": None, "max_VCI": None,
            })
            continue

        # Monthly composites
        mean_img = coll.mean().clip(aoi)
        min_img = coll.min().clip(aoi)
        max_img = coll.max().clip(aoi)

        # Server-side reduction (single feature = mean over AOI)
        def reduce_band(img, band):
            val = img.select(band).reduceRegion(
                reducer=ee.Reducer.mean(),
                geometry=aoi.geometry() if isinstance(aoi, ee.FeatureCollection) else aoi,
                scale=scale,
                maxPixels=1e13
            ).get(band)
            return val.getInfo() if val else None

        # Compute and append results
        rows.append({
            "month": m,
            "mean_NDVI": reduce_band(mean_img, "NDVI"),
            "min_NDVI": reduce_band(min_img, "NDVI"),
            "max_NDVI": reduce_band(max_img, "NDVI"),
            "mean_EVI": reduce_band(mean_img, "EVI"),
            "min_EVI": reduce_band(min_img, "EVI"),
            "max_EVI": reduce_band(max_img, "EVI"),
            "mean_VCI": reduce_band(mean_img, "VCI"),
            "min_VCI": reduce_band(min_img, "VCI"),
            "max_VCI": reduce_band(max_img, "VCI"),
        })

    # Convert to DataFrame
    df = pd.DataFrame(rows)
    print("✅ Finished computing monthly statistics.")
    display(df.head())

    # Save results
    os.makedirs("outputs", exist_ok=True)
    csv_path = f"outputs/sentinel2_monthly_summary_chunked_{year}.csv"
    df.to_csv(csv_path, index=False)
    print("📁 Saved CSV:", csv_path)

    return df


# --- Example usage ---
df_monthly = compute_monthly_stats_chunked(s2_indices, aoi, year=2024, scale=250)


📅 Computing monthly NDVI/EVI/VCI stats for 2024 (memory-safe)...
→ Month 01: 30 images
→ Month 02: 30 images
→ Month 03: 30 images
→ Month 04: 30 images
→ Month 05: 30 images
→ Month 06: 30 images
→ Month 07: 40 images
→ Month 08: 31 images
→ Month 09: 30 images
→ Month 10: 30 images
→ Month 11: 30 images
→ Month 12: 25 images
✅ Finished computing monthly statistics.


,month,mean_NDVI,min_NDVI,max_NDVI,mean_EVI,min_EVI,max_EVI,mean_VCI,min_VCI,max_VCI
0,1,0.421081,0.186669,0.596618,0.270755,0.049009,0.524337,0.546251,0.218014,0.795320
1,2,0.261660,0.179369,0.322142,0.491019,0.106086,2.211117,0.319211,0.203093,0.404578
2,3,0.226177,0.153709,0.278359,0.135770,0.104608,0.164059,0.269061,0.166740,0.343037
3,4,0.271759,0.231941,0.311247,0.231102,0.196302,0.266026,0.335782,0.280854,0.390317
4,5,0.573562,0.390320,0.721212,0.514717,0.379028,0.664273,0.758810,0.500392,0.967354


📁 Saved CSV: outputs/sentinel2_monthly_summary_chunked_2024.csv


In [ ]:
# Cell 6 — monthly composites function and extract stats per month over AOI
def compute_monthly_stats(collection, aoi, year=year, scale=10):
    """
    For each month, compute mean/min/max composites and then compute AOI mean
    of NDVI, EVI, VCI for each stat. Returns pandas DataFrame.
    """
    rows = []
    for m in range(1, 13):
        start = ee.Date.fromYMD(year, m, 1)
        end = start.advance(1, 'month')
        coll = collection.filterDate(start, end)

        # If no images, record NAs
        count = coll.size().getInfo()
        if count == 0:
            rows.append({'month': m, 'mean_NDVI': None, 'min_NDVI': None, 'max_NDVI': None,
                         'mean_EVI': None, 'min_EVI': None, 'max_EVI': None,
                         'mean_VCI': None, 'min_VCI': None, 'max_VCI': None})
            continue

        mean_img = coll.mean().clip(aoi)
        min_img = coll.min().clip(aoi)
        max_img = coll.max().clip(aoi)

        def aoi_mean(img, band):
            val = img.select(band).reduceRegion(
                reducer=ee.Reducer.mean(),
                geometry=aoi.geometry() if isinstance(aoi, ee.FeatureCollection) else aoi,
                scale=scale,
                maxPixels=1e13
            ).get(band)
            return None if val is None else val.getInfo()

        rows.append({
            'month': m,
            'mean_NDVI': aoi_mean(mean_img, 'NDVI'),
            'min_NDVI': aoi_mean(min_img, 'NDVI'),
            'max_NDVI': aoi_mean(max_img, 'NDVI'),
            'mean_EVI': aoi_mean(mean_img, 'EVI'),
            'min_EVI': aoi_mean(min_img, 'EVI'),
            'max_EVI': aoi_mean(max_img, 'EVI'),
            'mean_VCI': aoi_mean(mean_img, 'VCI'),
            'min_VCI': aoi_mean(min_img, 'VCI'),
            'max_VCI': aoi_mean(max_img, 'VCI'),
            'count': count
        })

    df = pd.DataFrame(rows)
    print("Computed monthly AOI stats DataFrame")
    display(df.head())
    return df

df_monthly = compute_monthly_stats(s2_indices, aoi, year=year, scale=10)
# save CSV
os.makedirs("outputs", exist_ok=True)
csv_path = f"outputs/sentinel2_monthly_summary_{year}.csv"
df_monthly.to_csv(csv_path, index=False)
print("Saved CSV:", csv_path)


In [10]:
# Cell 7 — interactive linked maps for NDVI/EVI/VCI with month selector
import ipywidgets as widgets
from IPython.display import display, clear_output

vis = {
    'NDVI': {'min': -0.2, 'max': 1.0, 'palette': ['FFFFFF','CE7E45','FCD163','99B718','207401','004C00']},
    'EVI' : {'min': -0.2, 'max': 1.0, 'palette': ['FFFFFF','ECE7F2','74A9CF','0570B0','023858']},
    'VCI' : {'min': 0.0, 'max': 1.0, 'palette': ['red','orange','yellow','lightgreen','green']}
}

month_selector = widgets.IntSlider(value=1, min=1, max=12, step=1, description='Month')

out = widgets.Output()

def update_maps(month):
    with out:
        clear_output(wait=True)
        start = ee.Date.fromYMD(year, month, 1)
        end = start.advance(1, 'month')
        monthly = s2_indices.filterDate(start, end).mean().clip(aoi)

        # Create three maps and add layers
        m1 = geemap.Map(center=aoi.geometry().centroid().coordinates().getInfo()[::-1], zoom=8, height='400px')
        m2 = geemap.Map(center=aoi.geometry().centroid().coordinates().getInfo()[::-1], zoom=8, height='400px')
        m3 = geemap.Map(center=aoi.geometry().centroid().coordinates().getInfo()[::-1], zoom=8, height='400px')

        m1.addLayer(monthly.select('NDVI'), vis['NDVI'], f"NDVI {year}-{month:02d}")
        m2.addLayer(monthly.select('EVI'), vis['EVI'], f"EVI {year}-{month:02d}")
        m3.addLayer(monthly.select('VCI'), vis['VCI'], f"VCI {year}-{month:02d}")

        for mm in (m1, m2, m3):
            mm.addLayer(aoi, {"color":"red"}, "AOI")
            mm.add_colorbar_branca(colors=vis['NDVI']['palette'] if mm is m1 else (vis['EVI']['palette'] if mm is m2 else vis['VCI']['palette']),
                                   vmin=(vis['NDVI']['min'] if mm is m1 else (vis['EVI']['min'] if mm is m2 else vis['VCI']['min'])),
                                   vmax=(vis['NDVI']['max'] if mm is m1 else (vis['EVI']['max'] if mm is m2 else vis['VCI']['max'])),
                                   caption='Index')

        linked = geemap.linked_maps(maps=[m1, m2, m3], labels=['NDVI','EVI','VCI'], rows=1, cols=3, height='450px')
        display(linked)

widgets.interact(update_maps, month=month_selector)
display(out)


interactive(children=(IntSlider(value=1, description='Month', max=12, min=1), Output()), _dom_classes=('widget…

Output()

In [11]:
# Cell 8 — function to create 12-month grid PNG for chosen index & stat (saves in outputs/)
import io
def save_monthly_grid_png(collection, aoi, index='NDVI', stat='mean', year=year, out_dir='outputs'):
    os.makedirs(out_dir, exist_ok=True)
    fig, axes = plt.subplots(3, 4, figsize=(16, 10))
    axes = axes.flatten()

    for m in range(1,13):
        ax = axes[m-1]
        start = ee.Date.fromYMD(year, m, 1)
        end = start.advance(1,'month')

        if stat == 'mean':
            img = collection.filterDate(start, end).mean().select(index).clip(aoi)
        elif stat == 'min':
            img = collection.filterDate(start, end).min().select(index).clip(aoi)
        elif stat == 'max':
            img = collection.filterDate(start, end).max().select(index).clip(aoi)
        else:
            raise ValueError("stat must be 'mean','min', or 'max'")

        # generate thumbnail and plot
        try:
            url = img.getThumbURL({
                "region": aoi.geometry(),
                "min": vis[index]['min'],
                "max": vis[index]['max'],
                "palette": vis[index]['palette'],
                "dimensions": 400
            })
            image_bytes = urllib.request.urlopen(url).read()
            im = Image.open(io.BytesIO(image_bytes))
            ax.imshow(np.array(im))
        except Exception as e:
            ax.text(0.5, 0.5, "No data", ha='center', va='center')
        ax.set_title(f"{index} {year}-{m:02d}")
        ax.axis('off')

    plt.suptitle(f"{index} ({stat}) monthly composites - {year}", fontsize=16)
    plt.tight_layout(rect=[0,0,1,0.96])
    outfile = os.path.join(out_dir, f"{index}_{stat}_{year}.png")
    plt.savefig(outfile, dpi=300, bbox_inches='tight')
    plt.close()
    print("Saved:", outfile)

# Example usage:
save_monthly_grid_png(s2_indices, aoi, index='NDVI', stat='mean', year=year)
save_monthly_grid_png(s2_indices, aoi, index='NDVI', stat='min', year=year)
save_monthly_grid_png(s2_indices, aoi, index='NDVI', stat='max', year=year)


Saved: outputs\NDVI_mean_2024.png
Saved: outputs\NDVI_min_2024.png
Saved: outputs\NDVI_max_2024.png


In [ ]:
# Cell 9 — create trend charts and save all artifacts in structured folder
def save_trend_plots_and_csv(df, out_dir='outputs', year=year):
    os.makedirs(out_dir, exist_ok=True)
    trends_dir = os.path.join(out_dir, 'trends'); os.makedirs(trends_dir, exist_ok=True)

    # Ensure df has the columns
    # df has mean_*, min_*, max_* columns created earlier
    # We'll make per-index DF and plot
    for index in ['NDVI','EVI','VCI']:
        col_mean = f"mean_{index}"
        col_min = f"min_{index}"
        col_max = f"max_{index}"
        if col_mean not in df.columns:
            print(f"No data columns for {index}, skipping.")
            continue
        d = df[['month', col_min, col_mean, col_max]].rename(columns={col_min:'min', col_mean:'mean', col_max:'max'})
        d = d.dropna(subset=['mean'], how='all')
        if d.empty:
            print(f"No values for {index}, skipping.")
            continue

        plt.figure(figsize=(10,5))
        plt.fill_between(d['month'], d['min'], d['max'], color='lightblue', alpha=0.3, label='min-max range')
        plt.plot(d['month'], d['mean'], marker='o', label='mean', color='navy')
        plt.title(f"{index} monthly trend {year}")
        plt.xlabel('Month')
        plt.xticks(range(1,13), ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"])
        plt.grid(alpha=0.3)
        plt.legend()
        outpng = os.path.join(trends_dir, f"{index}_trend_{year}.png")
        plt.savefig(outpng, dpi=300, bbox_inches='tight')
        plt.close()
        print("Saved trend:", outpng)

    # save CSV (df already saved earlier, but save to outputs folder)
    csv_out = os.path.join(out_dir, f"sentinel2_monthly_summary_{year}.csv")
    df.to_csv(csv_out, index=False)
    print("Saved CSV summary:", csv_out)

save_trend_plots_and_csv(df_monthly, out_dir='outputs', year=year)


In [ ]:
# Cell 10 — generate AOI map (thumbnail) and AOI info, then build PDF (requires reportlab)
from reportlab.platypus import SimpleDocTemplate, Paragraph, Image as RLImage, Spacer, Table, TableStyle, PageBreak
from reportlab.lib.pagesizes import A4, landscape
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet

def generate_aoi_image_and_info(aoi, out_dir='outputs'):
    os.makedirs(out_dir, exist_ok=True)
    mapfile = os.path.join(out_dir, 'aoi_map.png')
    infofile = os.path.join(out_dir, 'aoi_info.json')

    # geometry
    geom = aoi.geometry() if isinstance(aoi, ee.FeatureCollection) else aoi
    centroid = geom.centroid().coordinates().getInfo()
    bounds = geom.bounds().coordinates().getInfo()
    area_km2 = geom.area().getInfo() / 1e6

    # small thumbnail of AOI: create a simple RGB from Sentinel median for context
    try:
        rgb = (ee.ImageCollection('COPERNICUS/S2_SR')
               .filterBounds(aoi)
               .filterDate(start_date, end_date)
               .map(mask_s2_sr_clouds)
               .map(scale_s2)
               .select(['B4','B3','B2'])
               .median()
               .clip(geom)
               .visualize(min=0, max=0.3, bands=['B4','B3','B2']))
        url = rgb.getThumbURL({'region': geom, 'dimensions': 800})
        image_bytes = urllib.request.urlopen(url).read()
        Image.open(io.BytesIO(image_bytes)).save(mapfile)
    except Exception as e:
        print("Could not create AOI thumbnail:", e)

    info = {'centroid': {'lon':centroid[0], 'lat':centroid[1]},
            'area_km2': round(area_km2,2),
            'bounds': bounds}
    with open(infofile,'w') as f:
        json.dump(info,f,indent=2)

    print("AOI image & info saved:", mapfile, infofile)
    return mapfile, infofile

def create_pdf(output_dir='outputs', year=year):
    pdf_path = os.path.join(output_dir, f"Sentinel2_Report_{year}.pdf")
    doc = SimpleDocTemplate(pdf_path, pagesize=landscape(A4))
    styles = getSampleStyleSheet()
    elements = []

    # Title
    elements.append(Paragraph(f"Sentinel-2 Vegetation Report ({year})", styles['Title']))
    elements.append(Spacer(1,12))

    # AOI map & info
    mapfile, infofile = generate_aoi_image_and_info(aoi, out_dir=output_dir)
    if os.path.exists(mapfile):
        elements.append(Paragraph("Study Area", styles['Heading2']))
        elements.append(RLImage(mapfile, width=400, height=300))
        elements.append(Spacer(1,12))
    if os.path.exists(infofile):
        with open(infofile,'r') as f:
            info=json.load(f)
        data = [["Property","Value"],
                ["Area (km²)", f"{info['area_km2']}"],
                ["Centroid (Lat, Lon)", f"{info['centroid']['lat']:.4f}, {info['centroid']['lon']:.4f}"]]
        table=Table(data)
        table.setStyle(TableStyle([('GRID',(0,0),(-1,-1),0.25,colors.grey),('BACKGROUND',(0,0),(-1,0),colors.lightblue)]))
        elements.append(table)
        elements.append(PageBreak())

    # Add composite images and trend images (if present)
    comps = os.path.join(output_dir,'composites')
    trends = os.path.join(output_dir,'trends')

    # add composite files if they exist
    for index in ['NDVI','EVI','VCI']:
        for method in ['mean','min','max']:
            filepath = os.path.join(output_dir, f"{index}_{method}_{year}.png")
            if os.path.exists(filepath):
                elements.append(Paragraph(f"{index} - {method}", styles['Heading3']))
                elements.append(RLImage(filepath, width=700, height=350))
                elements.append(Spacer(1,12))
        elements.append(PageBreak())

    # add trend images
    elements.append(Paragraph("Trends", styles['Heading2']))
    for index in ['NDVI','EVI','VCI']:
        filepath = os.path.join(output_dir,'trends', f"{index}_trend_{year}.png")
        if os.path.exists(filepath):
            elements.append(Paragraph(index, styles['Heading3']))
            elements.append(RLImage(filepath, width=700, height=350))
            elements.append(Spacer(1,12))

    doc.build(elements)
    print("PDF created:", pdf_path)

# run
create_pdf(output_dir='outputs', year=year)
